<a href="https://colab.research.google.com/github/PatrickCalorioCarvalho/CA015ICMineracaoDeDados202601/blob/main/StrokePrediction_Preprocessamento_Sklearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

In [ ]:
# Carregar a base de dados
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             5110 non-null   object 
 1   age                5110 non-null   float64
 2   hypertension       5110 non-null   int64  
 3   heart_disease      5110 non-null   int64  
 4   ever_married       5110 non-null   object 
 5   work_type          5110 non-null   object 
 6   Residence_type     5110 non-null   object 
 7   avg_glucose_level  5110 non-null   float64
 8   bmi                4909 non-null   float64
 9   smoking_status     5110 non-null   object 
 10  stroke             5110 non-null   int64  
dtypes: float64(3), int64(3), object(5)
memory usage: 439.3+ KB


In [ ]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [ ]:
# Ajustes gerais
# i) Remover o atributo id
df = df.drop(columns=["id"])

In [ ]:
# ii) Separar atributos e classes
X = df.drop(columns=['stroke'])
y = df["stroke"]

In [ ]:
# iii) Guardar as colunas numéricas e categóricas
numeric_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

In [ ]:
# iv)Tratar os valores faltantes (SimpleImputer)
# Substituir NaN pela média em BMI
imputer_num = SimpleImputer(strategy="mean")
X[numeric_cols] = imputer_num.fit_transform(X[numeric_cols])

In [ ]:
X[numeric_cols].isna().sum()

,0
age,0
hypertension,0
heart_disease,0
avg_glucose_level,0
bmi,0


In [ ]:
# Análise génrica consiste em também aplicar
# SimpleImputer nos dados categóricos
#  utilizando a abordagem "most_frequent"

In [ ]:
# v) Normalizar os dados
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

In [ ]:
X[numeric_cols].head()

,age,hypertension,heart_disease,avg_glucose_level,bmi
0,1.051434,-0.328602,4.185032,2.706375,1.001234e+00
1,0.786070,-0.328602,-0.238947,2.121559,4.615554e-16
2,1.626390,-0.328602,4.185032,-0.005028,4.685773e-01
3,0.255342,-0.328602,-0.238947,1.437358,7.154182e-01
4,1.582163,3.043196,-0.238947,1.501184,-6.357112e-01


In [ ]:
# vi) Transformar as colunas categóricas com OHE
encoder = OneHotEncoder(sparse_output=False)
# Retorna em formato array
X_cat_encoder = encoder.fit_transform(X[categorical_cols])
# Seleciona o nome das colunas geradas
cat_names = encoder.get_feature_names_out(categorical_cols)
# Transformar em um Dataframe
X_cat_encoder = pd.DataFrame(
    X_cat_encoder,
    columns = cat_names,
    index= X.index
)

In [ ]:
# Ajustar o dataframe de atributos, removendo as features
# categoricas orginais e concatear com as novas features cat
X = X.drop(columns=categorical_cols)
X = pd.concat([X, X_cat_encoder], axis=1)

In [ ]:
X.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,1.051434,-0.328602,4.185032,2.706375,1.001234e+00,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,0.786070,-0.328602,-0.238947,2.121559,4.615554e-16,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1.626390,-0.328602,4.185032,-0.005028,4.685773e-01,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,0.255342,-0.328602,-0.238947,1.437358,7.154182e-01,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,1.582163,3.043196,-0.238947,1.501184,-6.357112e-01,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
X.shape, df.shape

((5110, 21), (5110, 11))

In [ ]:
# vii) Amostragem (holdout)
# Treino e teste (70/30 ou 80/20)
# train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.30,
    random_state= 2,
    stratify = y
)

In [ ]:
df["stroke"].value_counts()

,count
stroke,
0,4861
1,249


In [ ]:
X_train.shape, X_test.shape

((3577, 21), (1533, 21))

In [ ]:
y_train.shape, y_test.shape

((3577,), (1533,))

In [ ]:
# viii) treinar o modelo
# DecisionTree
modelo = DecisionTreeClassifier()
modelo.fit(X_train, y_train)

DecisionTreeClassifier()

In [ ]:
# ix) Avaliar modelo
y_pred = modelo.predict(X_test)

In [ ]:
y_test

,stroke
4557,0
3212,0
2427,0
2326,0
739,0
...,...
4669,0
1746,0
4533,0
1661,0


In [ ]:
y_pred

array([0, 0, 0, ..., 0, 1, 0])

In [ ]:
# Apresenta os dados no resumo classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.93      0.95      1458
           1       0.16      0.24      0.19        75

    accuracy                           0.90      1533
   macro avg       0.56      0.59      0.57      1533
weighted avg       0.92      0.90      0.91      1533



Atividade (Data Leakage):

Separar no inciio o X_train e X_test
* Scaler em X_train(fit_transform) e X_test (tranform)
* Imputer, OHE,
* Validar com o DecisionTree